# BlazingStar public federal-budget data — access tour

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abigailhaddad/blazingstar-data/blob/main/demo.ipynb)

[data.blazingstaranalytics.com](https://data.blazingstaranalytics.com/) is a free, **no-login, CC0 public-domain** mirror of the primary documents of federal spending and rulemaking, maintained by [BlazingStar Analytics](https://www.blazingstaranalytics.com/). It mirrors OMB MAX.gov, GovInfo/GPO, the Federal Register, and Regulations.gov.

Each dataset is a single `index.json` served from a Cloudflare CDN. **No API key, no auth.** Every record links back to its federal `source_url`; the execution data also carries a published SHA-256 hash for byte-level verification.

This notebook fetches each dataset into a pandas DataFrame and verifies one file against its hash. It is unofficial and not affiliated with BlazingStar Analytics.

In [ ]:
# In Colab this installs deps; locally it's a no-op if already installed.
%pip install -q requests pandas openpyxl

In [ ]:
import hashlib
from io import BytesIO

import pandas as pd
import requests

# The CDN rejects requests with no User-Agent, so use a session that sets one.
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "blazingstar-data-demo (https://github.com/abigailhaddad/blazingstar-data)"})


def get_json(url):
    """Fetch a JSON endpoint and return the parsed object."""
    r = SESSION.get(url, timeout=60)
    r.raise_for_status()
    return r.json()


def get_bytes(url):
    """Fetch a raw file (xlsx, pdf, mirrored json) as bytes."""
    r = SESSION.get(url, timeout=120)
    r.raise_for_status()
    return r.content

## 1. Apportionments (SF-132)

OMB's apportionment schedules — the legal limits on how agencies may spend, keyed by **TAFS** (Treasury Account Symbol). One index per fiscal year, FY2022–FY2026, ~10k rows each. Updated nightly.

Endpoint: `https://cdn.bzstr.co/sf132/fy{fy}/index.json`

In [ ]:
appt = get_json("https://cdn.bzstr.co/sf132/fy2026/index.json")
print(f"{appt['dataset']} — FY{appt['fiscal_year']}")
print(f"authoritative source: OMB MAX.gov (apportionment-public.max.gov)")
print(f"mirror generated_at:  {appt['generated_at']}  |  rows: {appt['file_count']}")

appt_df = pd.DataFrame(appt["files"])
# Each apportionment iteration is published as two rows (a data_json and a
# data_xlsx); only the data_json row carries the dollar amount. Keep those,
# and drop the BUREAU-* rows, which are bureau-level rollups (they would
# double-count against the individual accounts they aggregate).
appt_df = appt_df[(appt_df["file_type"] == "data_json")
                  & (~appt_df["tafs"].str.startswith("BUREAU", na=False))].copy()
appt_df["amount"] = pd.to_numeric(appt_df["total_approved_amount"], errors="coerce")
appt_df["iteration"] = pd.to_numeric(appt_df["iteration"], errors="coerce")
appt_df["approved"] = pd.to_datetime(appt_df["approval_timestamp"], utc=True, errors="coerce")
print("amount-bearing account rows:", len(appt_df), "| distinct TAFS:", appt_df["tafs"].nunique())

# Each row keeps its own source_url back to the OMB SF-132 it was mirrored from.
appt_df[["agency", "tafs", "account_name", "iteration", "amount", "approved", "source_url"]].head()

In [ ]:
# Current apportionment level for each account = its latest iteration.
latest = appt_df.sort_values("iteration").groupby(["agency", "tafs"], as_index=False).tail(1)

(
    latest.sort_values("amount", ascending=False)
    .loc[:, ["agency", "account_name", "tafs", "amount", "source_url"]]
    .head(10)
    .reset_index(drop=True)
)

## 3. Byte-level verification (SHA-256)

Each apportionment row carries a `mirror_url` (BlazingStar's copy) and a `hash_sha256`. Download the file and confirm the bytes match the published hash — this is the "don't take our word for it" guarantee. The `source_url` on the same row points to the original on OMB MAX.gov.

In [ ]:
# Prior-iteration amount for each account, then the latest row per account.
appt_df = appt_df.sort_values(["agency", "tafs", "iteration"])
appt_df["prev_amount"] = appt_df.groupby(["agency", "tafs"])["amount"].shift(1)
current = appt_df.groupby(["agency", "tafs"]).tail(1).copy()
current["delta"] = current["amount"] - current["prev_amount"]

anchor = appt_df["approved"].max()
window_start = anchor - pd.Timedelta(days=7)
changed = current[(current["approved"] >= window_start)
                  & current["delta"].notna() & (current["delta"] != 0)].copy()
changed = changed.reindex(changed["delta"].abs().sort_values(ascending=False).index)

print(f"Window: {window_start.date()} → {anchor.date()} (anchored to latest approval in file)")
print(f"Accounts with a changed apportionment: {len(changed)}")
print(f"Net change: ${changed['delta'].sum():,.0f}   |   Gross moved: ${changed['delta'].abs().sum():,.0f}")

show = changed[["agency", "account_name", "tafs", "prev_amount", "amount", "delta", "source_url"]].head(15)
show.style.format({"prev_amount": "${:,.0f}", "amount": "${:,.0f}", "delta": "${:+,.0f}"})

In [ ]:
## 4. SF-133 Execution

OMB's monthly Report on Budget Execution — the raw spreadsheets, one per agency per period (submitter-identity columns blanked, financial data byte-identical to OMB). Browsable by agency and fiscal year.

Index: `https://liatris.blazingstaranalytics.com/sf133/index.json`

sf133 = get_json("https://liatris.blazingstaranalytics.com/sf133/index.json")
print("source:    ", sf133["source"])
print("disclaimer:", sf133["disclaimer"][:140], "...")
print("agencies:", sf133["agency_count"], "| files:", sf133["file_count"])

# Each agency lists the fiscal years available and the latest file per year.
agency = sf133["agencies"][0]
latest = agency["years"][0]["latest"]
print("\nagency: ", agency["human_name"])
print("period: ", latest["period_label"], "FY", latest["fiscal_year"])
print("file:   ", latest["filename"], f"({latest['size']:,} bytes)")
print("source: ", latest["source_url"])

In [ ]:
row = appt_df[appt_df["mirror_url"].notna() & appt_df["hash_sha256"].notna()].iloc[0]
raw = get_bytes(row["mirror_url"])
computed = hashlib.sha256(raw).hexdigest()

print("file:    ", row["filename"])
print("bytes:   ", len(raw))
print("expected:", row["hash_sha256"])
print("computed:", computed)
print("MATCH:   ", computed == row["hash_sha256"])
print("source:  ", row["source_url"])

## 5. President's Budget Appendix

The machine-readable President's Budget Appendix — Program & Financing schedules, object classification, and employment for every federal account, as structured JSON. One index per fiscal year listing 29 volumes.

Index: `https://cdn.bzstr.co/budget_appendix/{fy}/json/index.json`

In [ ]:
appendix = get_json("https://cdn.bzstr.co/budget_appendix/2027/json/index.json")
print("FY:     ", appendix["fiscal_year"])
print("source: ", appendix["source"])
print("license:", appendix["license"])
print("totals: ", appendix["totals"])

# Each volume has a json_url with the per-account detail (P&F, object class, employment).
vol_df = pd.DataFrame(appendix["volumes"])
vol_df[["access_id", "department", "section_number", "source_url", "json_url"]].head(10)

In [ ]:
## 6. CFR Redlines

Consequential Federal Register proposed rules that amend the CFR, redlined the morning they publish. Updated daily.

Index: `https://cdn.bzstr.co/redlines/index.json`

redlines = get_json("https://cdn.bzstr.co/redlines/index.json")
redlines_df = pd.DataFrame(redlines)
print("rules:", len(redlines_df), "| columns:", list(redlines_df.columns))
# source_url = the Federal Register original; redline_url = BlazingStar's side-by-side.
redlines_df[["doc_number", "agency", "title", "comments_close_on", "source_url", "redline_url"]].head()

In [ ]:
## 7. Spend Plans

The spend plans agencies file when OMB restricts their funds — public by court order, mirrored as PDFs and browsable by agency and year.

Index: `https://cdn.bzstr.co/spend-plans/index.json`

plans = get_json("https://cdn.bzstr.co/spend-plans/index.json")
plans_df = pd.DataFrame(plans)
print("spend-plan PDFs:", len(plans_df))
# url = the mirrored PDF; source_url = where it was filed (MAX.gov).
plans_df[["agency", "bureau", "fiscal_year", "filename", "url", "source_url"]].head()

In [ ]:
redlines = get_json("https://cdn.bzstr.co/redlines/index.json")
redlines_df = pd.DataFrame(redlines)
print("rules:", len(redlines_df), "| columns:", list(redlines_df.columns))
redlines_df[["doc_number", "agency", "title", "comments_close_on", "redline_url"]].head()

## How to cite

The data is **CC0 1.0 (public domain)** — no attribution is legally required — but for any analysis you publish, cite the **primary federal source**, which travels with every record, and note the mirror you actually pulled from:

> *[Agency] [account / TAFS], [dataset], OMB MAX.gov / GovInfo / Federal Register, retrieved via BlazingStar Analytics' public mirror (data.blazingstaranalytics.com, CC0), [date].*

Every row in this notebook exposes a `source_url` pointing at the authoritative origin. For the apportionment and SF-133 figures specifically, the mirror publishes a **SHA-256** you can recompute (Section 3 above) — so a citation can assert byte-equivalence with OMB, not just "we downloaded a copy." When provenance matters, lead with the `source_url` and treat the BlazingStar mirror as the access path.

## 6. Spend Plans

The spend plans agencies file when OMB restricts their funds — public by court order, mirrored as PDFs and browsable by agency and year.

Index: `https://cdn.bzstr.co/spend-plans/index.json`

In [ ]:
plans = get_json("https://cdn.bzstr.co/spend-plans/index.json")
plans_df = pd.DataFrame(plans)
print("spend-plan PDFs:", len(plans_df))
plans_df[["agency", "bureau", "fiscal_year", "filename", "url"]].head()

## The join key: TAFS

The **Treasury Account Symbol (TAFS)** ties these datasets together and out to [USAspending](https://www.usaspending.gov/): an apportionment (SF-132) sets the legal limit on a TAFS, the SF-133 reports execution against it, and awards in USAspending draw down from it — appropriation → apportionment → execution → award on one identifier.

All data here is **CC0 1.0 (public domain)**, mirrored from federal sources, each record stamped with its origin `source_url`. See [data.blazingstaranalytics.com](https://data.blazingstaranalytics.com/).